In [32]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [33]:
df = pd.read_csv('ecommerceDataset4.csv')
df = df.rename(columns={'Unnamed: 0': 'id'})
df

,id,category,text
0,32612,Clothing & Accessories,Jainsons Women's Belt This product is made fro...
1,15861,Household,Velkro Krevia Anti Theft Motion Sensor Alarm L...
2,48978,Electronics,Tewtross Bluetooth Transmitter and Receiver fo...
3,12119,Household,"Krosno Europe Non-Lead Crystal-Clear Glass, Vi..."
4,368,Household,Ilu Dream Catcher Wall Hanging Handmade Beaded...
...,...,...,...
3146,10960,Household,"Chef's Garage 3 Stage Kitchen Knife Sharpener,..."
3147,15561,Household,FAVY LAURELS Room HALOGEN HEATER with 3 Heatin...
3148,38182,Clothing & Accessories,SKiDlers Women's Cotton Maternity Top Impress ...
3149,13551,Household,Amazing Stainless Steel Pasta Maker Noodle Mak...


In [34]:
stoplist = stopwords.words('english')
lk = ['n', 'v', 'a', 'r', 's']

def preprocess_text(line):
    line = re.sub(r'[^\w\s\'’-]', '', line.strip())
    clean_line = " ".join([word for word in line.split() if word.lower() not in stoplist])
    
    lemmatizer = WordNetLemmatizer()
    words = word_tokenize(clean_line)
    pos_tagged = nltk.pos_tag(words)
    lemmatized_words = []

    for i, word in enumerate(words):
        word_pos = pos_tagged[i][1].lower() if pos_tagged[i][1].lower() in lk else 'n'
        lemmatized_words.append(lemmatizer.lemmatize(word.lower(), pos=word_pos))
    
    return " ".join(lemmatized_words)

In [35]:
df['processed_text'] = df['text'].astype(str).apply(preprocess_text)
df

,id,category,text,processed_text
0,32612,Clothing & Accessories,Jainsons Women's Belt This product is made fro...,jainsons woman 's belt product made waxed cott...
1,15861,Household,Velkro Krevia Anti Theft Motion Sensor Alarm L...,velkro krevia anti theft motion sensor alarm l...
2,48978,Electronics,Tewtross Bluetooth Transmitter and Receiver fo...,tewtross bluetooth transmitter receiver tv hom...
3,12119,Household,"Krosno Europe Non-Lead Crystal-Clear Glass, Vi...",krosno europe non-lead crystal-clear glass viv...
4,368,Household,Ilu Dream Catcher Wall Hanging Handmade Beaded...,ilu dream catcher wall hanging handmade beaded...
...,...,...,...,...
3146,10960,Household,"Chef's Garage 3 Stage Kitchen Knife Sharpener,...",chef 's garage 3 stage kitchen knife sharpener...
3147,15561,Household,FAVY LAURELS Room HALOGEN HEATER with 3 Heatin...,favy laurel room halogen heater 3 heating elem...
3148,38182,Clothing & Accessories,SKiDlers Women's Cotton Maternity Top Impress ...,skidlers woman 's cotton maternity top impress...
3149,13551,Household,Amazing Stainless Steel Pasta Maker Noodle Mak...,amazing stainless steel pasta maker noodle mak...


In [36]:
df["category"].value_counts()

category
Household                 1207
Books                      739
Electronics                663
Clothing & Accessories     542
Name: count, dtype: int64

In [37]:
df_sampled = df.groupby("category").apply(lambda x: x.sample(n=542, random_state=0))
df_sampled = df_sampled.reset_index(drop=True)
df_sampled["category"].value_counts()

category
Books                     542
Clothing & Accessories    542
Electronics               542
Household                 542
Name: count, dtype: int64

In [38]:
from sklearn.model_selection import train_test_split

train_corpus, test_corpus, train_label_names, test_label_names = train_test_split(df_sampled['processed_text'], df_sampled['category'], test_size=0.3, random_state=0)
train_corpus.shape, test_corpus.shape

((1517,), (651,))

In [39]:
from sklearn.feature_extraction.text import CountVectorizer

# Сумка біграм
bv = CountVectorizer(ngram_range=(2,2))
train_corpus_bv = bv.fit_transform(train_corpus)
test_corpus_bv = bv.transform(test_corpus)

In [40]:
from sklearn.naive_bayes import MultinomialNB

# Наївний байєсів класифікатор
mnb = MultinomialNB()
mnb.fit(train_corpus_bv, train_label_names)

MultinomialNB()

In [41]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

predictions = mnb.predict(test_corpus_bv)
accuracy = accuracy_score(test_label_names, predictions)
print('Accuracy:', round(accuracy, 4))
print('Classification report:')
print(classification_report(test_label_names, predictions))
print('Confusion matrix:')
print(confusion_matrix(test_label_names, predictions))

Accuracy: 0.8618
Classification report:
                        precision    recall  f1-score   support

                 Books       0.95      0.77      0.85       170
Clothing & Accessories       0.89      0.93      0.91       161
           Electronics       0.90      0.90      0.90       163
             Household       0.73      0.85      0.79       157

              accuracy                           0.86       651
             macro avg       0.87      0.86      0.86       651
          weighted avg       0.87      0.86      0.86       651

Confusion matrix:
[[131   5   4  30]
 [  0 149   3   9]
 [  1   5 147  10]
 [  6   8   9 134]]


In [42]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

# GridSearchCV
mnb_pipeline = Pipeline([('bv', CountVectorizer(ngram_range=(2,2))),
                         ('mnb', MultinomialNB())])
param_grid = {
    'mnb__alpha': [1e-3, 1e-2, 1e-1, 1.0, 10.0],
    'mnb__fit_prior': [True, False]
}

gs_mnb = GridSearchCV(mnb_pipeline, param_grid, cv=5)
gs_mnb.fit(train_corpus, train_label_names)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('bv',
                                        CountVectorizer(ngram_range=(2, 2))),
                                       ('mnb', MultinomialNB())]),
             param_grid={'mnb__alpha': [0.001, 0.01, 0.1, 1.0, 10.0],
                         'mnb__fit_prior': [True, False]})

In [43]:
print("Best Parameters:", gs_mnb.best_params_)

Best Parameters: {'mnb__alpha': 1.0, 'mnb__fit_prior': False}


In [44]:
mnb_bv = MultinomialNB(alpha=gs_mnb.best_params_['mnb__alpha'],
                       fit_prior=gs_mnb.best_params_['mnb__fit_prior'])
mnb_bv.fit(train_corpus_bv, train_label_names)

predictions = mnb_bv.predict(test_corpus_bv)
accuracy = accuracy_score(test_label_names, predictions)
print('Accuracy:', round(accuracy, 4))
print('Classification report:')
print(classification_report(test_label_names, predictions))
print('Confusion matrix:')
print(confusion_matrix(test_label_names, predictions))

Accuracy: 0.8879
Classification report:
                        precision    recall  f1-score   support

                 Books       0.89      0.90      0.89       170
Clothing & Accessories       0.89      0.93      0.91       161
           Electronics       0.90      0.90      0.90       163
             Household       0.87      0.82      0.84       157

              accuracy                           0.89       651
             macro avg       0.89      0.89      0.89       651
          weighted avg       0.89      0.89      0.89       651

Confusion matrix:
[[153   5   4   8]
 [  6 149   3   3]
 [  2   5 147   9]
 [ 11   8   9 129]]


In [45]:
from sklearn.svm import LinearSVC

# Опорні вектори
svm = LinearSVC(random_state=0, max_iter=2000)
svm.fit(train_corpus_bv, train_label_names)

LinearSVC(max_iter=2000, random_state=0)

In [46]:
predictions = svm.predict(test_corpus_bv)
accuracy = accuracy_score(test_label_names, predictions)
print('Accuracy:', round(accuracy, 4))
print('Classification report:')
print(classification_report(test_label_names, predictions))
print('Confusion matrix:')
print(confusion_matrix(test_label_names, predictions))

Accuracy: 0.6882
Classification report:
                        precision    recall  f1-score   support

                 Books       0.47      0.99      0.64       170
Clothing & Accessories       0.97      0.70      0.82       161
           Electronics       0.98      0.56      0.72       163
             Household       0.90      0.47      0.62       157

              accuracy                           0.69       651
             macro avg       0.83      0.68      0.70       651
          weighted avg       0.83      0.69      0.70       651

Confusion matrix:
[[169   1   0   0]
 [ 45 113   1   2]
 [ 65   0  92   6]
 [ 80   2   1  74]]


In [47]:
# GridSearchCV
svm_pipeline = Pipeline([('bv', CountVectorizer(ngram_range=(2,2))),
                         ('svm', LinearSVC(random_state=0, max_iter=2000))])
param_grid = {
    'svm__fit_intercept': [True, False],
    'svm__C': np.logspace(-2, 2, 5),
    'svm__loss': ['squared_hinge']
}

gs_svm = GridSearchCV(svm_pipeline, param_grid, cv=3)
gs_svm.fit(train_corpus, train_label_names)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('bv',
                                        CountVectorizer(ngram_range=(2, 2))),
                                       ('svm',
                                        LinearSVC(max_iter=2000,
                                                  random_state=0))]),
             param_grid={'svm__C': array([1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02]),
                         'svm__fit_intercept': [True, False],
                         'svm__loss': ['squared_hinge']})

In [48]:
print("Best Parameters:", gs_svm.best_params_)

Best Parameters: {'svm__C': 0.01, 'svm__fit_intercept': False, 'svm__loss': 'squared_hinge'}


In [49]:
svm_bv = LinearSVC(random_state=0,
                   max_iter=2000,
                   fit_intercept=gs_svm.best_params_['svm__fit_intercept'],
                   C=gs_svm.best_params_['svm__C'],
                   loss=gs_svm.best_params_['svm__loss'])
svm_bv.fit(train_corpus_bv, train_label_names)

predictions = svm_bv.predict(test_corpus_bv)
accuracy = accuracy_score(test_label_names, predictions)
print('Accuracy:', round(accuracy, 4))
print('Classification report:')
print(classification_report(test_label_names, predictions))
print('Confusion matrix:')
print(confusion_matrix(test_label_names, predictions))

Accuracy: 0.8817
Classification report:
                        precision    recall  f1-score   support

                 Books       0.88      0.92      0.90       170
Clothing & Accessories       0.88      0.92      0.90       161
           Electronics       0.93      0.88      0.90       163
             Household       0.84      0.81      0.82       157

              accuracy                           0.88       651
             macro avg       0.88      0.88      0.88       651
          weighted avg       0.88      0.88      0.88       651

Confusion matrix:
[[156   5   2   7]
 [  6 148   2   5]
 [  3   5 143  12]
 [ 12  11   7 127]]


In [50]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF 
tv = TfidfVectorizer(ngram_range=(2,2))
train_corpus_tv = tv.fit_transform(train_corpus)
test_corpus_tv = tv.transform(test_corpus)

In [51]:
# Наївний байєсів класифікатор
mnb = MultinomialNB()
mnb.fit(train_corpus_tv, train_label_names)

MultinomialNB()

In [52]:
predictions = mnb.predict(test_corpus_tv)
accuracy = accuracy_score(test_label_names, predictions)
print('Accuracy:', round(accuracy, 4))
print('Classification report:')
print(classification_report(test_label_names, predictions))
print('Confusion matrix:')
print(confusion_matrix(test_label_names, predictions))

Accuracy: 0.8648
Classification report:
                        precision    recall  f1-score   support

                 Books       0.96      0.78      0.86       170
Clothing & Accessories       0.89      0.93      0.91       161
           Electronics       0.92      0.90      0.91       163
             Household       0.73      0.86      0.79       157

              accuracy                           0.86       651
             macro avg       0.87      0.87      0.87       651
          weighted avg       0.88      0.86      0.87       651

Confusion matrix:
[[133   5   3  29]
 [  0 149   2  10]
 [  1   5 146  11]
 [  5   9   8 135]]


In [53]:
# GridSearchCV
mnb_pipeline = Pipeline([('tv', TfidfVectorizer(ngram_range=(2,2))),
                         ('mnb', MultinomialNB())])
param_grid = {
    'mnb__alpha': [1e-3, 1e-2, 1e-1, 1.0, 10.0],
    'mnb__fit_prior': [True, False]
}

gs_mnb = GridSearchCV(mnb_pipeline, param_grid, cv=5)
gs_mnb.fit(train_corpus, train_label_names)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tv',
                                        TfidfVectorizer(ngram_range=(2, 2))),
                                       ('mnb', MultinomialNB())]),
             param_grid={'mnb__alpha': [0.001, 0.01, 0.1, 1.0, 10.0],
                         'mnb__fit_prior': [True, False]})

In [54]:
print("Best Parameters:", gs_mnb.best_params_)

Best Parameters: {'mnb__alpha': 0.001, 'mnb__fit_prior': False}


In [55]:
mnb_tv = MultinomialNB(alpha=gs_mnb.best_params_['mnb__alpha'],
                       fit_prior=gs_mnb.best_params_['mnb__fit_prior'])
mnb_tv.fit(train_corpus_tv, train_label_names)

predictions = mnb_tv.predict(test_corpus_tv)
accuracy = accuracy_score(test_label_names, predictions)
print('Accuracy:', round(accuracy, 4))
print('Classification report:')
print(classification_report(test_label_names, predictions))
print('Confusion matrix:')
print(confusion_matrix(test_label_names, predictions))

Accuracy: 0.8894
Classification report:
                        precision    recall  f1-score   support

                 Books       0.89      0.91      0.90       170
Clothing & Accessories       0.90      0.94      0.92       161
           Electronics       0.90      0.90      0.90       163
             Household       0.87      0.81      0.84       157

              accuracy                           0.89       651
             macro avg       0.89      0.89      0.89       651
          weighted avg       0.89      0.89      0.89       651

Confusion matrix:
[[154   6   4   6]
 [  6 151   2   2]
 [  2   3 147  11]
 [ 11   8  11 127]]


In [56]:
# Опорні вектори
svm = LinearSVC(random_state=0, max_iter=2000)
svm.fit(train_corpus_tv, train_label_names)

LinearSVC(max_iter=2000, random_state=0)

In [57]:
predictions = svm.predict(test_corpus_tv)
accuracy = accuracy_score(test_label_names, predictions)
print('Accuracy:', round(accuracy, 4))
print('Classification report:')
print(classification_report(test_label_names, predictions))
print('Confusion matrix:')
print(confusion_matrix(test_label_names, predictions))

Accuracy: 0.8879
Classification report:
                        precision    recall  f1-score   support

                 Books       0.87      0.92      0.90       170
Clothing & Accessories       0.91      0.93      0.92       161
           Electronics       0.92      0.89      0.91       163
             Household       0.85      0.81      0.83       157

              accuracy                           0.89       651
             macro avg       0.89      0.89      0.89       651
          weighted avg       0.89      0.89      0.89       651

Confusion matrix:
[[157   4   2   7]
 [  6 149   2   4]
 [  4   2 145  12]
 [ 13   9   8 127]]


In [58]:
# GridSearchCV
svm_pipeline = Pipeline([('tv', TfidfVectorizer(ngram_range=(2,2))),
                         ('svm', LinearSVC(random_state=0, max_iter=2000))])
param_grid = {
    'svm__fit_intercept': [True, False],
    'svm__C': np.logspace(-2, 2, 5),
    'svm__loss': ['squared_hinge']
}

gs_svm = GridSearchCV(svm_pipeline, param_grid, cv=3)
gs_svm.fit(train_corpus, train_label_names)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('tv',
                                        TfidfVectorizer(ngram_range=(2, 2))),
                                       ('svm',
                                        LinearSVC(max_iter=2000,
                                                  random_state=0))]),
             param_grid={'svm__C': array([1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02]),
                         'svm__fit_intercept': [True, False],
                         'svm__loss': ['squared_hinge']})

In [59]:
print("Best Parameters:", gs_svm.best_params_)

Best Parameters: {'svm__C': 1.0, 'svm__fit_intercept': True, 'svm__loss': 'squared_hinge'}


In [60]:
svm_tv = LinearSVC(random_state=0,
                   max_iter=2000,
                   fit_intercept=gs_svm.best_params_['svm__fit_intercept'],
                   C=gs_svm.best_params_['svm__C'],
                   loss=gs_svm.best_params_['svm__loss'])
svm_tv.fit(train_corpus_tv, train_label_names)

predictions = svm_tv.predict(test_corpus_tv)
accuracy = accuracy_score(test_label_names, predictions)
print('Accuracy:', round(accuracy, 4))
print('Classification report:')
print(classification_report(test_label_names, predictions))
print('Confusion matrix:')
print(confusion_matrix(test_label_names, predictions))

Accuracy: 0.8879
Classification report:
                        precision    recall  f1-score   support

                 Books       0.87      0.92      0.90       170
Clothing & Accessories       0.91      0.93      0.92       161
           Electronics       0.92      0.89      0.91       163
             Household       0.85      0.81      0.83       157

              accuracy                           0.89       651
             macro avg       0.89      0.89      0.89       651
          weighted avg       0.89      0.89      0.89       651

Confusion matrix:
[[157   4   2   7]
 [  6 149   2   4]
 [  4   2 145  12]
 [ 13   9   8 127]]
